# Step1: Source injection

In [ ]:
import lib.butler as btl
import lib.tools as tl
import lib.stamp as stp
import lib.inject as inj
import lib.visual as vis

from astropy.io import fits
#import os
from pathlib import Path

import random
import json
import glob

import lsst.afw.image as afwImage

In [ ]:
#%matplotlib inline

In [ ]:
folder = Path(tl.CATALOG_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

folder = Path(tl.FIG_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

Begin with ECDFS (center). 

In [ ]:
ra_cen = 53.182
dec_cen = -28.104

#low galactic
#ra_cen = 95.0
#dec_cen = -25.0

#low ecliptic
#ra_cen = 37.98
#dec_cen = 7.015

BAND = tl.BAND

Define the grid for injection. It is based on RA/DEC rather than X/Y pixel coordinates. 

Also, the grid is near the (ECDFS) field center; try to match the image center.

In [ ]:
LENS_INDEX_MIN = tl.LENS_INDEX_MIN    # first batch: 0; next batch: 1024; etc. (must match step0)
N_LENS = tl.N_LENS

WIDTH_ARCMIN = tl.WIDTH_ARCMIN 
NUM_SIDE = tl.NUM_SIDE

inj_radec = inj.make_grid(ra_cen, dec_cen, WIDTH_ARCMIN, NUM_SIDE)

## Visit Image Injection

In [ ]:
dataset_refs = btl.get_visit_dataset_refs(ra_cen, dec_cen, BAND)
print(len(dataset_refs))
print(dataset_refs[0])

### Select one visit

In [ ]:
n_visits = tl.BAND_EXPOSURE_TOTAL[BAND]
visit_index = random.randint(0, n_visits - 1)
print(f"Randomly selected visit_index: {visit_index}")

visit_image = btl.get_visit_image(dataset_refs, visit_index)
visit_rotation_angle = tl.get_visit_rotation_angle(visit_image)
print(visit_rotation_angle, "deg")

In [ ]:
dataId = dataset_refs[visit_index].dataId
visit = dataId['visit']
detector = dataId['detector']
visit_tag = f"visit_{visit}_{detector}"

### Choose a stamp (and rotate), collect point source data

In [ ]:
stamp_folder = tl.STAMP_FOLDER

Rotate the static stamp before injecting into the visit image. Also collect point source mags and coords.

In [ ]:
stamp_filename_list = []
stamp_mag_list = []
ind_list = []
time_indices = []
system_indices = []

for ind in range(N_LENS):

    system_index = LENS_INDEX_MIN + ind
    if system_index % 100 == 0:
        print(f"system_index: {system_index}")

    # Use the static stamp
    wcs_stamp_filename = f"{stamp_folder}/system_{system_index}_static_{BAND}_wcs.fits"
    p = Path(wcs_stamp_filename)
    if not p.exists():
        continue

    # Check if coordinate is within the visit image
    ra = inj_radec[ind]['ra']
    dec = inj_radec[ind]['dec']
    wcs_visit = visit_image.getWcs()
    x, y = wcs_visit.skyToPixelArray(ra, dec, degrees=True)
    x, y = x[0], y[0]
    bbox = visit_image.getBBox()
    if x < bbox.beginX or x > bbox.endX or y < bbox.beginY or y > bbox.endY:
        continue

    # Rotate static stamp
    rot_wcs_stamp_filename = wcs_stamp_filename.replace(".fits", f"_rot_{visit_tag}.fits")
    stamp_filename_list.append(rot_wcs_stamp_filename)

    stamp_mag = fits.getval(wcs_stamp_filename, "TOT_MAG")
    stamp_mag_list.append(stamp_mag)

    path = Path(rot_wcs_stamp_filename)
    if not path.exists():
        stp.make_rotated_stamp(visit_rotation_angle, wcs_stamp_filename, rot_wcs_stamp_filename)

    # Pick a random epoch for point source mags
    n_obs = stp.get_n_observations(system_index, BAND, stp.LENS_FILENAMES[0])
    time_indices.append(random.randint(0, n_obs - 1))
    system_indices.append(system_index)
    ind_list.append(ind)

print(f"Number of systems to inject: {len(ind_list)}")

# Collect point source data
point_delta_x, point_delta_y = stp.get_all_point_delta_coords(system_indices, BAND)
point_mags_visit = stp.get_all_point_mags_visit(system_indices, time_indices, BAND)

### Inject the (rotated) stamp to a visit image

Inject onto the grid.

In [ ]:
#inj_radec

In [ ]:
wcs_visit = visit_image.getWcs()
tag = f"visit_{visit}_{detector}"

inj_catalog = inj.make_inj_catalog(
    wcs_visit, stamp_mag_list, stamp_filename_list,
    inj_radec[ind_list],
    point_mags_visit, point_delta_x, point_delta_y,
    visit_rotation_angle, tag
)

injected_visit_image, injected_visit_catalog = inj.visit_inject(visit_image, inj_catalog)

visit_image.writeFits(f"{tl.FIG_FOLDER}/{visit_tag}.fits")
injected_visit_image.writeFits(f"{tl.FIG_FOLDER}/injected_{visit_tag}.fits")

In [ ]:
#injected_visit_catalog

In [ ]:
# They have injected, not needed
for stamp_filename in stamp_filename_list:
    p = Path(stamp_filename)
    if p.exists():
        p.unlink()

### Check the image with plot

Make a plot at the central region.

In [ ]:
vis.plot_dual(visit_image, injected_visit_image, inj_radec, visit_tag)

## Template injection

In [ ]:
template_filename_list = glob.glob(f"{tl.FIG_FOLDER}/template_*_*_{BAND}.fits")
print(template_filename_list)

if len(template_filename_list)==0:

    dataset_refs = btl.get_template_dataset_refs(ra_cen, dec_cen, BAND)
    print(len(dataset_refs))
    print(dataset_refs[0])


### Select one template

In [ ]:
if len(template_filename_list)==0:
    
    template_index = 0 
    template_image = btl.get_template_image(dataset_refs, template_index)

    dataId = dataset_refs[template_index].dataId
    tract = dataId['tract']
    patch = dataId['patch']
    
else:
    tmp = template_filename_list[0].split('/')[-1].split('_')
    tract = int(tmp[1])
    patch = int(tmp[2])

template_tag = f"template_{tract}_{patch}_{BAND}"

In [ ]:
#template_image.getWcs()
#xc = template_image.getX0() + template_image.getWidth() / 2.
#yc = template_image.getY0() + template_image.getHeight() / 2.
#print(wcs.pixelToSky(xc, yc))

### Choose a stamp, collect point source data (mean mags)

In [ ]:
if len(template_filename_list) == 0:

    static_stamp_filename_list = []
    static_stamp_mag_list = []
    template_ind_list = []
    template_system_indices = []

    for ind in range(N_LENS):
        system_index = LENS_INDEX_MIN + ind

        wcs_stamp_filename = f"{stamp_folder}/system_{system_index}_static_{BAND}_wcs.fits"
        p = Path(wcs_stamp_filename)
        if not p.exists():
            continue

        static_stamp_filename_list.append(wcs_stamp_filename)
        static_stamp_mag = fits.getval(wcs_stamp_filename, "TOT_MAG")
        static_stamp_mag_list.append(static_stamp_mag)
        template_ind_list.append(ind)
        template_system_indices.append(system_index)

    print(f"Number of systems for template: {len(template_ind_list)}")

    # Collect point source data with mean mags
    point_delta_x_t, point_delta_y_t = stp.get_all_point_delta_coords(template_system_indices, BAND)
    point_mags_template = stp.get_all_point_mags_template(template_system_indices, BAND)

### Inject the (unrotated) stamp to a template image

Inject onto the grid.

In [ ]:
template_filename = f"{tl.FIG_FOLDER}/{template_tag}.fits"
injected_template_filename = f"{tl.FIG_FOLDER}/injected_{template_tag}.fits"

if len(template_filename_list) == 0:

    wcs_template = template_image.getWcs()

    inj_catalog_template = inj.make_inj_catalog(
        wcs_template, static_stamp_mag_list, static_stamp_filename_list,
        inj_radec[template_ind_list],
        point_mags_template, point_delta_x_t, point_delta_y_t,
        0, template_tag
    )

    injected_template_image, injected_template_catalog = inj.template_inject(template_image, inj_catalog_template)

    template_image.writeFits(template_filename)
    injected_template_image.writeFits(injected_template_filename)

else:
    template_image = afwImage.ExposureF.readFits(template_filename)
    injected_template_image = afwImage.ExposureF.readFits(injected_template_filename)

In [ ]:
#injected_template_catalog

### Check the image with plot

Make a plot at the central region.

In [ ]:
vis.plot_dual(template_image, injected_template_image, inj_radec, template_tag, )

In [ ]:
print(visit_tag)
print(template_tag)

In [ ]:
data = {
    "visit": visit,
    "detector": detector,
    "tract": tract,
    "patch": patch,
    "band": BAND,
}

with open("numbers.json", "w") as f:
    json.dump(data, f)

